# 11_f Download Player-Season Batting Stats

Purpose: download annual MLB player-season hitting totals so BA / OPS / OBP / SLG can be evaluated at player-season level.

Runtime: CPU/network. This does not download videos or train models.

Outputs:
- `raw_player_stats/mlb_statsapi_hitting_{season}.json`
- `manifests/player_season_batting_v1.parquet`
- `reports/preflight/player_season_batting_stats_v1.json`

Next: `23_player_season_aggregate_baseline.ipynb` can use these labels for BA / OPS / OBP / SLG.


In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates: list[Path] = []
    env_root = os.environ.get('BATTING_CODE_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            fixed,
            Path.cwd(),
        ]
    )
    candidates.extend(Path.cwd().parents)

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            if candidate != fixed:
                print('WARNING: fixed REPO_DIR ではない場所の repo を使います。')
                print('  fixed:', fixed)
                print('  using:', candidate)
                print('  次回は repo フォルダを fixed path に置くか、BATTING_CODE_ROOT を明示してください。')
            return candidate

    raise FileNotFoundError(
        'sport_pipeline package が見つかりません。Drive には notebook 単体ではなく repo フォルダごと置く必要があります。\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別の場所に置いた場合は、この notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/batting_codex_handoff\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()

REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(
        'sport_pipeline を import できません。repo 配置と src/sport_pipeline/__init__.py を確認してください。\n'
        f'REPO_DIR={REPO_DIR}\n'
        f'src_dir={src_dir}\n'
        f'src_dir_exists={src_dir.exists()}'
    )

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('CACHE_DIR =', CACHE_DIR)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('src_dir =', src_dir)

from sport_pipeline.pipeline.run_profile import load_run_profile, resolve_statcast_date_range, run_id, stage_settings, threshold

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
RUN_ID = run_id(RUN_PROFILE, 'full_run_id')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('RUN_ID =', RUN_ID)


In [ ]:
from sport_pipeline.statcast.player_season_stats import download_player_season_batting_stats

PLAYER_STATS_SETTINGS = stage_settings(RUN_PROFILE, 'player_season_stats')
ENABLE_DOWNLOAD = bool(PLAYER_STATS_SETTINGS.get('execute_default', True))
SEASONS = PLAYER_STATS_SETTINGS.get('seasons') or RUN_PROFILE.get('data_window', {}).get('expected_seasons')
TIMEOUT_SEC = int(PLAYER_STATS_SETTINGS.get('timeout_sec', 60))
FORCE = bool(PLAYER_STATS_SETTINGS.get('force', False))

print('ENABLE_DOWNLOAD =', ENABLE_DOWNLOAD)
print('SEASONS =', SEASONS)
print('TIMEOUT_SEC =', TIMEOUT_SEC, 'FORCE =', FORCE)

outputs = download_player_season_batting_stats(
    BASE_DIR,
    seasons=[int(season) for season in SEASONS] if SEASONS else None,
    execute=ENABLE_DOWNLOAD,
    timeout_sec=TIMEOUT_SEC,
    force=FORCE,
)
for name, path in outputs.items():
    print(name, '->', path, 'exists=', path.exists())


In [ ]:
summary = json.loads(outputs['summary'].read_text(encoding='utf-8')) if outputs['summary'].exists() else {}
print(json.dumps(summary, ensure_ascii=False, indent=2)[:5000])
if ENABLE_DOWNLOAD and int(summary.get('manifest_rows') or 0) == 0:
    raise RuntimeError('player-season batting stats manifest is empty')
print('NEXT: notebooks/23_player_season_aggregate_baseline.ipynb')
